# 🆚 Claude vs GPT-4 비교 분석

## 개요
- **목적**: Claude와 GPT-4 실험 결과를 직접 비교 분석
- **입력**: `experiments.csv` (Claude) + `experiments_gpt.csv` (GPT-4)
- **기능**: 통계 비교, 시각화, 통계 검정, 비교 보고서 생성

## 사용 방법
1. ✅ 패키지 설치
2. ✅ 라이브러리 import
3. 📁 두 CSV 파일 업로드
4. 📊 자동 비교 분석 실행
5. 💾 비교 보고서 다운로드

---
## 1️⃣ 패키지 설치

In [ ]:
# 필수 패키지 설치
!pip install pandas matplotlib seaborn numpy scipy -q

print("✅ 패키지 설치 완료")

---
## 2️⃣ 라이브러리 Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from collections import Counter
from google.colab import files
import json
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 스타일 설정
sns.set_style('whitegrid')
sns.set_palette('Set2')

print("✅ 라이브러리 import 완료")

---
## 3️⃣ CSV 파일 업로드 (Claude 결과)

**먼저 Claude 실험 결과를 업로드하세요:**
- 파일명: `experiments.csv` (또는 claude가 포함된 파일)

In [ ]:
# Claude 결과 업로드
print("📁 Claude 실험 결과 CSV 파일을 선택하세요...")
uploaded_claude = files.upload()

if len(uploaded_claude) == 0:
    print("❌ 파일이 업로드되지 않았습니다.")
else:
    claude_file = list(uploaded_claude.keys())[0]
    print(f"\n✅ Claude 파일 업로드 완료: {claude_file}")

---
## 4️⃣ CSV 파일 업로드 (GPT-4 결과)

**다음으로 GPT-4 실험 결과를 업로드하세요:**
- 파일명: `experiments_gpt.csv` (또는 gpt가 포함된 파일)

In [ ]:
# GPT-4 결과 업로드
print("📁 GPT-4 실험 결과 CSV 파일을 선택하세요...")
uploaded_gpt = files.upload()

if len(uploaded_gpt) == 0:
    print("❌ 파일이 업로드되지 않았습니다.")
else:
    gpt_file = list(uploaded_gpt.keys())[0]
    print(f"\n✅ GPT-4 파일 업로드 완료: {gpt_file}")

---
## 5️⃣ 데이터 로드 및 검증

In [ ]:
# 데이터 로드
try:
    df_claude = pd.read_csv(claude_file)
    df_gpt = pd.read_csv(gpt_file)
    
    print("✅ 데이터 로드 완료\n")
    print(f"Claude: {len(df_claude)}개 실험")
    print(f"GPT-4: {len(df_gpt)}개 실험")
    
    # API 구분 컬럼 추가
    df_claude['api'] = 'Claude'
    df_gpt['api'] = 'GPT-4'
    
    # 성공한 실험만 필터링
    df_claude_success = df_claude[df_claude['status'] == 'success'].copy()
    df_gpt_success = df_gpt[df_gpt['status'] == 'success'].copy()
    
    print(f"\nClaude 성공: {len(df_claude_success)}개")
    print(f"GPT-4 성공: {len(df_gpt_success)}개")
    
except Exception as e:
    print(f"❌ 파일 로드 실패: {e}")

---
## 6️⃣ 기본 통계 비교

In [ ]:
print("="*80)
print("📊 Claude vs GPT-4 비교 요약")
print("="*80)

# 성공률 비교
claude_total = len(df_claude)
claude_success = len(df_claude_success)
claude_success_rate = (claude_success / claude_total * 100) if claude_total > 0 else 0

gpt_total = len(df_gpt)
gpt_success = len(df_gpt_success)
gpt_success_rate = (gpt_success / gpt_total * 100) if gpt_total > 0 else 0

print(f"\n🎯 성공률 비교:")
print(f"   Claude:  {claude_success}/{claude_total} ({claude_success_rate:.1f}%)")
print(f"   GPT-4:   {gpt_success}/{gpt_total} ({gpt_success_rate:.1f}%)")
print(f"   차이:    {abs(claude_success_rate - gpt_success_rate):.1f}%p")

if len(df_claude_success) > 0 and len(df_gpt_success) > 0:
    # 점수 비교
    claude_mean = df_claude_success['total_score'].mean()
    claude_std = df_claude_success['total_score'].std()
    gpt_mean = df_gpt_success['total_score'].mean()
    gpt_std = df_gpt_success['total_score'].std()
    
    print(f"\n📈 평균 점수 비교:")
    print(f"   Claude:  {claude_mean:.2f} ± {claude_std:.2f}")
    print(f"   GPT-4:   {gpt_mean:.2f} ± {gpt_std:.2f}")
    print(f"   차이:    {abs(claude_mean - gpt_mean):.2f}점")
    
    # 실행 시간 비교
    claude_time = df_claude_success['execution_time_sec'].mean()
    gpt_time = df_gpt_success['execution_time_sec'].mean()
    
    print(f"\n⏱️ 평균 실행 시간:")
    print(f"   Claude:  {claude_time:.2f}초")
    print(f"   GPT-4:   {gpt_time:.2f}초")
    print(f"   차이:    {abs(claude_time - gpt_time):.2f}초")
    
    # 통계적 유의성 검정 (t-test)
    t_stat, p_value = stats.ttest_ind(
        df_claude_success['total_score'], 
        df_gpt_success['total_score']
    )
    
    print(f"\n📊 통계 검정 (Independent t-test):")
    print(f"   t-statistic: {t_stat:.4f}")
    print(f"   p-value:     {p_value:.4f}")
    
    if p_value < 0.05:
        print(f"   결과:        통계적으로 유의한 차이 있음 (p < 0.05) ✓")
        winner = "Claude" if claude_mean > gpt_mean else "GPT-4"
        print(f"   승자:        {winner}")
    else:
        print(f"   결과:        통계적으로 유의한 차이 없음 (p >= 0.05)")
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt((claude_std**2 + gpt_std**2) / 2)
    cohens_d = (claude_mean - gpt_mean) / pooled_std
    
    print(f"\n📏 효과 크기 (Cohen's d):")
    print(f"   d = {cohens_d:.4f}")
    if abs(cohens_d) < 0.2:
        effect = "작음 (small)"
    elif abs(cohens_d) < 0.5:
        effect = "중간 (medium)"
    else:
        effect = "큼 (large)"
    print(f"   해석:        {effect}")

print("\n" + "="*80)

---
## 7️⃣ 시각화 - 성공률 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Claude 성공률
sizes_claude = [claude_success, claude_total - claude_success]
colors = ['#3498db', '#e74c3c']
axes[0].pie(sizes_claude, labels=['Success', 'Failed'], colors=colors, 
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11, 'weight': 'bold'})
axes[0].set_title(f'Claude Success Rate\n{claude_success}/{claude_total} ({claude_success_rate:.1f}%)', 
                 fontsize=13, weight='bold', pad=15)

# GPT-4 성공률
sizes_gpt = [gpt_success, gpt_total - gpt_success]
axes[1].pie(sizes_gpt, labels=['Success', 'Failed'], colors=colors, 
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11, 'weight': 'bold'})
axes[1].set_title(f'GPT-4 Success Rate\n{gpt_success}/{gpt_total} ({gpt_success_rate:.1f}%)', 
                 fontsize=13, weight='bold', pad=15)

plt.suptitle('Success Rate Comparison', fontsize=15, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparison_success_rate.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 그래프 저장: comparison_success_rate.png")

---
## 8️⃣ 시각화 - 점수 분포 비교

In [ ]:
if len(df_claude_success) > 0 and len(df_gpt_success) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # 히스토그램 비교
    axes[0].hist(df_claude_success['total_score'], bins=20, alpha=0.6, 
                label='Claude', color='#3498db', edgecolor='black')
    axes[0].hist(df_gpt_success['total_score'], bins=20, alpha=0.6, 
                label='GPT-4', color='#e67e22', edgecolor='black')
    axes[0].axvline(claude_mean, color='#3498db', linestyle='--', linewidth=2, 
                   label=f'Claude Mean: {claude_mean:.2f}')
    axes[0].axvline(gpt_mean, color='#e67e22', linestyle='--', linewidth=2, 
                   label=f'GPT-4 Mean: {gpt_mean:.2f}')
    axes[0].set_xlabel('Total Score', fontsize=12, weight='bold')
    axes[0].set_ylabel('Frequency', fontsize=12, weight='bold')
    axes[0].set_title('Score Distribution Comparison', fontsize=14, weight='bold')
    axes[0].legend(loc='upper left', fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # 박스플롯 비교
    data_to_plot = [df_claude_success['total_score'], df_gpt_success['total_score']]
    bp = axes[1].boxplot(data_to_plot, labels=['Claude', 'GPT-4'], 
                         patch_artist=True, widths=0.6)
    bp['boxes'][0].set_facecolor('#3498db')
    bp['boxes'][1].set_facecolor('#e67e22')
    for box in bp['boxes']:
        box.set_alpha(0.7)
    for median in bp['medians']:
        median.set_color('red')
        median.set_linewidth(2)
    
    axes[1].set_ylabel('Total Score', fontsize=12, weight='bold')
    axes[1].set_title('Score Box Plot Comparison', fontsize=14, weight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # 통계 정보 추가
    stats_text = f"Claude: {claude_mean:.2f}±{claude_std:.2f}\n"
    stats_text += f"GPT-4: {gpt_mean:.2f}±{gpt_std:.2f}\n"
    stats_text += f"p-value: {p_value:.4f}"
    axes[1].text(0.02, 0.98, stats_text, transform=axes[1].transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig('comparison_score_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ 그래프 저장: comparison_score_distribution.png")
else:
    print("⚠️ 성공한 실험이 부족하여 점수 비교를 그릴 수 없습니다.")

---
## 9️⃣ 시각화 - 조합별 성능 비교

In [ ]:
if len(df_claude_success) > 0 and len(df_gpt_success) > 0:
    # 조합별 평균 점수
    claude_by_comb = df_claude_success.groupby('combination_id')['total_score'].mean()
    gpt_by_comb = df_gpt_success.groupby('combination_id')['total_score'].mean()
    
    # 공통 조합만 선택
    common_combs = sorted(set(claude_by_comb.index) & set(gpt_by_comb.index))
    
    if len(common_combs) > 0:
        claude_scores = [claude_by_comb[c] for c in common_combs]
        gpt_scores = [gpt_by_comb[c] for c in common_combs]
        
        # 조합 이름 매핑
        comb_names = df_claude_success.groupby('combination_id')['combination_name'].first().to_dict()
        labels = [f"{c}\n{comb_names.get(c, '')}" for c in common_combs]
        
        fig, ax = plt.subplots(figsize=(16, 10))
        
        x = np.arange(len(common_combs))
        width = 0.35
        
        bars1 = ax.barh(x - width/2, claude_scores, width, label='Claude', 
                       color='#3498db', alpha=0.8, edgecolor='black')
        bars2 = ax.barh(x + width/2, gpt_scores, width, label='GPT-4', 
                       color='#e67e22', alpha=0.8, edgecolor='black')
        
        ax.set_yticks(x)
        ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlabel('Average Total Score', fontsize=12, weight='bold')
        ax.set_title('Performance Comparison by Algorithm Combination', 
                    fontsize=14, weight='bold', pad=20)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3, axis='x')
        
        # 점수 값 표시
        for i, (c_score, g_score) in enumerate(zip(claude_scores, gpt_scores)):
            ax.text(c_score + 0.5, i - width/2, f"{c_score:.1f}", 
                   va='center', fontsize=8, weight='bold')
            ax.text(g_score + 0.5, i + width/2, f"{g_score:.1f}", 
                   va='center', fontsize=8, weight='bold')
        
        plt.tight_layout()
        plt.savefig('comparison_by_combination.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("✅ 그래프 저장: comparison_by_combination.png")
    else:
        print("⚠️ 공통 조합이 없어 비교할 수 없습니다.")
else:
    print("⚠️ 성공한 실험이 부족하여 조합별 비교를 그릴 수 없습니다.")

---
## 🔟 시각화 - 대화 유형별 성능 비교

In [ ]:
if len(df_claude_success) > 0 and len(df_gpt_success) > 0:
    # 대화 유형별 평균 점수
    claude_by_type = df_claude_success.groupby('conversation_type')['total_score'].mean()
    gpt_by_type = df_gpt_success.groupby('conversation_type')['total_score'].mean()
    
    # 공통 대화 유형
    common_types = sorted(set(claude_by_type.index) & set(gpt_by_type.index))
    
    if len(common_types) > 0:
        claude_type_scores = [claude_by_type[t] for t in common_types]
        gpt_type_scores = [gpt_by_type[t] for t in common_types]
        
        fig, ax = plt.subplots(figsize=(12, 7))
        
        x = np.arange(len(common_types))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, claude_type_scores, width, label='Claude', 
                      color='#3498db', alpha=0.8, edgecolor='black')
        bars2 = ax.bar(x + width/2, gpt_type_scores, width, label='GPT-4', 
                      color='#e67e22', alpha=0.8, edgecolor='black')
        
        ax.set_xticks(x)
        ax.set_xticklabels(common_types, rotation=45, ha='right')
        ax.set_ylabel('Average Total Score', fontsize=12, weight='bold')
        ax.set_title('Performance Comparison by Conversation Type', 
                    fontsize=14, weight='bold', pad=20)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
        
        # 점수 값 표시
        for i, (c_score, g_score) in enumerate(zip(claude_type_scores, gpt_type_scores)):
            ax.text(i - width/2, c_score + 1, f"{c_score:.1f}", 
                   ha='center', fontsize=10, weight='bold')
            ax.text(i + width/2, g_score + 1, f"{g_score:.1f}", 
                   ha='center', fontsize=10, weight='bold')
        
        plt.tight_layout()
        plt.savefig('comparison_by_conversation_type.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("✅ 그래프 저장: comparison_by_conversation_type.png")
    else:
        print("⚠️ 공통 대화 유형이 없어 비교할 수 없습니다.")
else:
    print("⚠️ 성공한 실험이 부족하여 대화 유형별 비교를 그릴 수 없습니다.")

---
## 1️⃣1️⃣ 상세 통계 검정

In [ ]:
if len(df_claude_success) > 0 and len(df_gpt_success) > 0:
    print("="*80)
    print("📊 조합별 통계 검정 (Independent t-test)")
    print("="*80)
    
    results = []
    
    for comb in common_combs:
        claude_comb_scores = df_claude_success[df_claude_success['combination_id'] == comb]['total_score']
        gpt_comb_scores = df_gpt_success[df_gpt_success['combination_id'] == comb]['total_score']
        
        if len(claude_comb_scores) > 1 and len(gpt_comb_scores) > 1:
            t_stat, p_val = stats.ttest_ind(claude_comb_scores, gpt_comb_scores)
            
            results.append({
                'combination_id': comb,
                'combination_name': comb_names.get(comb, ''),
                'claude_mean': claude_comb_scores.mean(),
                'gpt_mean': gpt_comb_scores.mean(),
                'difference': claude_comb_scores.mean() - gpt_comb_scores.mean(),
                't_statistic': t_stat,
                'p_value': p_val,
                'significant': 'Yes' if p_val < 0.05 else 'No'
            })
    
    if len(results) > 0:
        results_df = pd.DataFrame(results)
        results_df = results_df.sort_values('p_value')
        
        print(f"\n조합별 비교 결과 (p-value 낮은 순):")
        print()
        
        for idx, row in results_df.head(10).iterrows():
            print(f"{row['combination_id']} ({row['combination_name']}):")
            print(f"  Claude: {row['claude_mean']:.2f}")
            print(f"  GPT-4:  {row['gpt_mean']:.2f}")
            print(f"  차이:   {row['difference']:.2f}")
            print(f"  p-value: {row['p_value']:.4f} {'✓ 유의함' if row['significant'] == 'Yes' else ''}")
            print()
        
        # CSV 저장
        results_df.to_csv('statistical_tests.csv', index=False)
        print("✅ 통계 검정 결과 저장: statistical_tests.csv")
    
    print("\n" + "="*80)
else:
    print("⚠️ 성공한 실험이 부족하여 통계 검정을 수행할 수 없습니다.")

---
## 1️⃣2️⃣ 비교 보고서 생성

In [ ]:
if len(df_claude_success) > 0 and len(df_gpt_success) > 0:
    report = f"""# Claude vs GPT-4 비교 분석 보고서

## 1. 실험 개요

- **Claude 실험**: {claude_total}개 (성공: {claude_success}개, {claude_success_rate:.1f}%)
- **GPT-4 실험**: {gpt_total}개 (성공: {gpt_success}개, {gpt_success_rate:.1f}%)

## 2. 성능 비교

### 2.1 평균 점수

| API | 평균 점수 | 표준편차 |
|-----|----------|----------|
| Claude | {claude_mean:.2f} | {claude_std:.2f} |
| GPT-4 | {gpt_mean:.2f} | {gpt_std:.2f} |
| **차이** | **{abs(claude_mean - gpt_mean):.2f}** | - |

### 2.2 통계 검정

- **검정 방법**: Independent t-test
- **t-statistic**: {t_stat:.4f}
- **p-value**: {p_value:.4f}
- **결과**: {'통계적으로 유의한 차이 있음 (p < 0.05)' if p_value < 0.05 else '통계적으로 유의한 차이 없음 (p >= 0.05)'}
- **Effect size (Cohen\'s d)**: {cohens_d:.4f} ({effect})

### 2.3 실행 시간

| API | 평균 실행 시간 |
|-----|---------------|
| Claude | {claude_time:.2f}초 |
| GPT-4 | {gpt_time:.2f}초 |
| **차이** | **{abs(claude_time - gpt_time):.2f}초** |

## 3. 결론

""";
    
    if p_value < 0.05:
        winner = "Claude" if claude_mean > gpt_mean else "GPT-4"
        report += f"""- **{winner}**가 통계적으로 유의하게 더 높은 성능을 보였습니다.
- 평균 점수 차이는 {abs(claude_mean - gpt_mean):.2f}점이며, 효과 크기는 {effect}입니다.
"""
    else:
        report += f"""- Claude와 GPT-4 간에 통계적으로 유의한 성능 차이가 없었습니다.
- 두 API 모두 유사한 수준의 마인드맵 생성 성능을 보였습니다.
"""
    
    report += f"""
## 4. 생성된 파일

- `comparison_success_rate.png` - 성공률 비교
- `comparison_score_distribution.png` - 점수 분포 비교
- `comparison_by_combination.png` - 조합별 성능 비교
- `comparison_by_conversation_type.png` - 대화 유형별 성능 비교
- `statistical_tests.csv` - 조합별 통계 검정 결과

---

**생성일**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""
    
    # 보고서 저장
    with open('comparison_report.md', 'w', encoding='utf-8') as f:
        f.write(report)
    
    print("✅ 비교 보고서 저장: comparison_report.md")
    print("\n" + "="*80)
    print(report)
    print("="*80)
else:
    print("⚠️ 성공한 실험이 부족하여 비교 보고서를 생성할 수 없습니다.")

---
## 1️⃣3️⃣ 모든 파일 다운로드

In [ ]:
import os

print("📦 생성된 파일 목록:\n")

output_files = [
    'comparison_success_rate.png',
    'comparison_score_distribution.png',
    'comparison_by_combination.png',
    'comparison_by_conversation_type.png',
    'statistical_tests.csv',
    'comparison_report.md'
]

existing_files = [f for f in output_files if os.path.exists(f)]

for f in existing_files:
    print(f"   ✅ {f}")

if len(existing_files) > 0:
    print(f"\n💾 {len(existing_files)}개 파일을 다운로드합니다...")
    for f in existing_files:
        files.download(f)
    print("\n✅ 다운로드 완료!")
else:
    print("\n⚠️ 다운로드할 파일이 없습니다.")

---
## 🎉 비교 분석 완료!

### 생성된 파일:
1. **그래프 (4개)**:
   - `comparison_success_rate.png` - 성공률 비교
   - `comparison_score_distribution.png` - 점수 분포 비교
   - `comparison_by_combination.png` - 조합별 성능 비교
   - `comparison_by_conversation_type.png` - 대화 유형별 성능 비교

2. **통계 자료 (2개)**:
   - `statistical_tests.csv` - 조합별 통계 검정 결과
   - `comparison_report.md` - 📊 비교 분석 보고서

### 다음 단계:
- 다운로드한 파일을 논문/보고서에 첨부
- `comparison_report.md`를 참고하여 결과 해석
- 통계적 유의성을 고려하여 결론 도출